# Visualization with Matplotlib 
This lesson has been adapted from a tutorial notebook from Thomas Caswell, the project lead of matplotlib. 

## Harmonic Oscillator Case Study
This is a short case-study of how to go from "raw" measurement (or simulation) data through exploratory visualization to an (almost) paper ready figure.
In this scenario, assume that you have fabricated (or simulated) 25 [cantilevers](https://en.wikipedia.org/wiki/Cantilever). 
There is some value (suggestively called "control") that varies between the cantilevers.  
To characterize the cantilever, you flex them by some fixed displacement and release the tip. You then record a time (in ms) series of the displacement (in mm) as the vibrations damp out.

The data for these exercises comes from a `.py` script called `gendata`. 

Let's start by importing some useful libraries. As well as the module that will generate the sample data we will be working with.

In [ ]:
# standard libraries
%matplotlib inline
import matplotlib.pyplot as plt 
# scipy libraries
import numpy as np
# custom module
import gendata as gt

We'll be using the function `get_data`, so let's use `help` to get a sneak preview of how we should use the function.

In [ ]:
help(gt.get_data)

So looking at this we can tell that the number of cantilevers is the first parameter and we should get out arrays of multiple dimensions. 
Since they're arrays, we can get a sense of the structure of their data from the `shape` attribute.

In [ ]:
# get data for N=25 cantilevers
d, time, control = gt.get_data(25) #displacement, time, control value for the corresponding experiment
print(d.shape)
print(time.shape)
print(control.shape)

So now we know that there are 4112 time points and 25 control values. Essentially, we have 4112 time points for 25 different cantilever experiments. The displacement data is measured for each cantilever experiment at each time point and each row corresponds to the time series for a single experiment. 


If we try to plot the data as is, because matplotlib assumes each column is a different line, we end up with a big mess (4112 lines)
If we plot the transpose of the data instead, we should end up with 25 lines for every column, which at least lines up with the time points.
(It is still kind of a big mess as they're all sitting on top of each other.)

In [ ]:
fig, ax = plt.subplots()
#ax.plot(d)
#ax.plot(time, d.T)

If we want to just grab one cantilever, we need to grab a single row. Let's grab the 7th experiment now.
If we do it correctly, we should be able to plot it against time (the shape should match the time array)
We can also use the same index to grab the corresponding control value of the experiment.

In [ ]:
d7 = d[6,:] # for 2D arrays: rows are 1st index, columns are 2nd index
c7 = control[6] # corresponding control value
print(d7.shape)
print(c7)

<h2><span class="fa fa-flash"></span> 1. Plot a single experiment</h2>

+ Write a function to plot the displacement vs. time for a single experiment. 
    The function should take an integer between 0 and 24 for each of the cantilever experiments and plot the corresponding line. The line should be labeled with the corresponding control value.
+ Make a test figure: test your function by plotting the 7th cantilever experiment within a figure environment with axes that show the corresponding x- and y- axis labels. 

In [ ]:
def plot_one(time, index):
    """
    Takes an array of time points for the x-axis and plots the displacement of an experiment indexed by "index" from the "d" array. 
    Labels with the corresponding control value.
    """
    #(ln,) = plt.plot(...., label=....)
    #return {"raw": ln}

f,ax = plt.subplots(1)
# plot_one(..,...)
# plt.legend()
ax.set_xlabel('time [ms]')
ax.set_ylabel('displacement [mm]')


<h2><span class="fa fa-flash"></span> 2. Plot Multiple Lines </h2>

Now, let's say we want to plot multiple lines:  
+ Make a figure: use a `for` loop and your function, make a figure like the one before now showing the results of the 1st, 6th, and last experiments all on the same plot.



In [ ]:
# your code here

This is still pretty cluttered and hard to see differences in the periods, for example.  Because we know that the displacements are a relative measurement from the equilibrium position, we can try to offset the curves vertically.
+ Write a different version of your plotting command that takes in the parameter `offset`, which indicates how far in the y-direction you can plot the line. (You can just add this value to the displacement array).  
    Use the provided annotation command to replace the label of the control value to appear above each line.
+ Make a figure: plot the same 3 lines as before, this time vertically offset from one another by different amounts so as to be evenly vertically spaced on the same plot.

In [ ]:
def plot_one_offset(time, index, offset):
    """
    Takes an array of time points for the x-axis and plots the displacement of an experiment indexed by "index" from the "d" array translated in y by the input value "offset"
    Labels with the corresponding control value with annotation.
    """
    ax = plt.gca() # grabs the assumed current axes environment

    #(ln,) = plt.plot(..,...)
    control_val = 0.0 # replace
    ann = ax.annotate(
        (
            f"$C={control_val:.1f}$\n"
        ),
        # units are (axes-fraction, data)
        xy=(0.95, offset + 0.5),
        xycoords=ax.get_yaxis_transform(),
        # set the text alignment
        ha="right",
        va="bottom",
    )
    #return {"raw": ln, "annotation":ann}

fig, ax = plt.subplots()
#for ...:
    #plot_one_offset(time, index, offset)

ax.set_xlabel("time [ms]")
ax.set_ylabel("displacement + offset [mm]")

In a real experiment, it's common to have to compare experimental data to a fit. In `get_data`, there is a defined function `fit` that fits the displacement to the following function of time:

$$
z(t) = A e^{-\zeta\omega_0t} \sin\left(\sqrt{1 - \zeta^2}\omega_0t + \varphi\right).
$$

by fitting for the parameters: $A, \zeta, \omega, \varphi$

In [ ]:
help(gt.fit)

Let's fit the 7th experiment again and use the `sample` method to see the fit.

In [ ]:
fitvals = gt.fit(d7,time)
print(fitvals)
print(fitvals.A, fitvals.zeta, fitvals.omega, fitvals.phi) # can access the fit values individually
f,ax = plt.subplots()
ax.plot(time,fitvals.sample(time))


<h2><span class="fa fa-flash"></span> 3. Plotting a Fit </h2>

+ Amend your plotting function from the previous part to take the resulting `fit_vals` and use the `sample` method to plot the fit for each experiment on top of the line.
+ Add the values of $\zeta$ and $\omega$ to the annotation.
+ Make a figure of the three lines offset from one another with the corresponding fits labeled.


In [ ]:
def plot_one_offset(time, index, fitvals, offset):
    """
    Takes an array of time points for the x-axis and plots the displacement of an experiment indexed by "index" from the "d" array translated in y by the input value "offset"
    Plots the corresponding fit overlaid on top as a black line.
    Labels with the corresponding control value with an annotation.
    """
    ax = plt.gca()
    
    #(ln,) = plt.plot(...,...)
    #(fit,) = plt.plot(...,...)
    ann = ax.annotate(
        (
            f"$C={0.0:.1f}$\n" # fill in with label values
            f"$\\zeta={0.0:.2f}, \\omega={0.0:.2f}$"
        ),
        # units are (axes-fraction, data)
        xy=(0.95, offset + 0.5),
        xycoords=ax.get_yaxis_transform(),
        # set the text alignment
        ha="right",
        va="bottom",
    )
    #return {"raw": ln, "fit":fit, "annotation":ann}

fig, ax = plt.subplots()

#for ....: 
    # fit_vals = gt.fit(...)
    # plot_one_offset(time, index, fit_vals,offset)


ax.set_xlabel("time [ms]")
ax.set_ylabel("displacement + offset [mm]")

From a science point of view, we want to actually look at how the fit parameters change with the control value in aggregate across all our experiments, so we can write a loop to see all the omega values, for example.

In [ ]:
for i, d_i in enumerate(d):
    this_fit = gt.fit(d_i,time)
    print(this_fit.omega)

In order to plot it later, we should just store the fits for all the parameters in our loop. Dictionaries to the rescue!

In [ ]:
N = d.shape[0] # total number of experiments

# define dictionary to save fit results
all_fits = {"A":np.zeros([N]),
            "zeta":np.zeros([N]),
            "omega":np.zeros([N]),
            "phi":np.zeros([N]),
            "control":control}

for i, m in enumerate(d):
    this_fit = gt.fit(m,time)
    all_fits["A"][i] = this_fit.A
    all_fits["zeta"][i] = this_fit.zeta
    all_fits["omega"][i] = this_fit.omega
    all_fits["phi"][i] = this_fit.phi


```{exercise} Check-in Question
:label: ex_1

How do you get the value of the "zeta" parameter for the 7th experiment from the `all_fits` dictionary? (You can check your guess with the value from the cell in the previous section.)
```

````{solution} ex_1
:class: dropdown

```{code-block} python
all_fits["zeta"][6]
```

````


<h2><span class="fa fa-flash"></span> 4. Showing multiple panels of data </h2>


Let's plot the correlations with `control`. (But these should really go on a separate axis from each other, otherwise it will look a mess again).
+ Write two functions: `plot_zeta` and `plot_omega` that take an input axis and plot $\zeta$ and $\omega$, respectively, against the control on each axis object. Label the x- and y-axes within the function.
    Plot $\zeta$ with markers only and $\omega$ with markers and lines.
+ Test your functions on the multiple axis figure environment. 

Which of the parameters are actually correlated with the value of the `control` variable?

In [ ]:
def plot_zeta(ax, all_fits):
    """
    With input:
    ax: axis to plot on
    all_fits: dictionary of all parameters
    """
    ax.set_ylabel(r"$\zeta$")
    ax.set_xlabel("control ")
    ax.set_ylim(0, 0.1)
    return #ax.plot(..., ..., marker=..., color="k",linestyle=...)


def plot_omega(ax, all_fits):
    ax.set_ylabel(r"$\omega_0/2\pi$ [kHz]")
    ax.set_xlabel("control ")
    ax.set_ylim(0, 1.5)
    return #ax.plot(..., ..., marker=...,color="k",linestyle=...)

fig, (ax1, ax2) = plt.subplots(2, 1, constrained_layout=True,sharex=True)
#plot_zeta(ax1, all_fits)
#plot_omega(ax2, all_fits)


Now let's say we want to add our actual experimental lines onto another axis of this same plot. But we wrote that code assuming it would all be on one axis!
The easy thing to do here is to wrap our code for making the plot of the lines and fits into a function that can show any number of lines all on the same axis.
+ Write a function to plot several lines that uses your function from the previous part and takes an input axis and a list of input indices.
+ Test it on the multipanel figure provided below using `subplot_mosaic` to set up differently sized panels

In [ ]:
def plot_several(ax,idx):
    """
    Inputs:
    ax: the axis object onto which we will plot multiple lines
    idx: list of indices of the experiments for which we will plot the lines for
    will plot as many lines as there are indices using the plot_one_offset(...) function
    """
    out = []
    # for...:
        # fit_vals = 
        #lines = plot_one_offset(time,index,fit_vals,offset=4*j)
        #out.append(lines)
    ax.set_xlabel("time [ms]")
    ax.set_ylabel("displacement [mm]")

    return out

plot_several(ax=plt.gca(),idx=[0,5,-1])

If all functions are defined properly, all you need is to press play here.

In [ ]:
single_col_width = 8.6 / 2.54  # single column APS figure
double_col_width = 17.8 / 2.54  # double column APS figure
fig, ax_dict = plt.subplot_mosaic(
    [["raw", "omega"], ["raw", "zeta"]], constrained_layout=True
)
fig.set_size_inches(double_col_width, double_col_width * 0.5)
indx = [0, 10, 24]
plot_several(ax_dict["raw"],indx)
plot_zeta(ax_dict["zeta"], all_fits)
plot_omega(ax_dict["omega"], all_fits)

fig.align_ylabels(list(ax_dict.values()))

#can even save this as a pdf
#fig.savefig("fig2.pdf")

## Conclusion

This has been a speed run from get-it-on-the-screen exploratory data analysis to designing a figure for publication.  Over the course of this notebook, we have slowly built up a mini-library of helper functions tuned to exactly this data set for this experiment. By building our functions out in a modular and hierarchical way, we have left room for us to go back and repurpose the bits and pieces into new types of figures and analyses. 

We have highlighted a small sample of useful matplotlib features:
+ legends and annotations
+ multiple panel figures with subplots
+ figure sizes and alignment

In practice, matplotlib has a lot more features that allow for a great deal of customizability and types of plots. 

The [matplotlib gallery](https://matplotlib.org/stable/gallery/index.html) has an organized index of example code for *hundreds* of potential use cases. If you want to make a certain of plot or feature, it's a good first stop to check out.